# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTD504/flyrank-ai-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**ML Task Type:** Binary Classification (or Refresh Priority Scoring)

**Why this task type:**
- The core objective of the Content Refresh lane is to identify which existing published articles require a refresh/re-optimization versus those that should be left alone.
- Framing this as Binary Classification (`needs_refresh`: `1` vs `0`) allows us to train a supervised model that flags high-risk/fading content.
- Alternatively, the output probability score from the classifier acts as a **Priority Score** ($0.0$ to $1.0$) to rank and queue candidate articles for editorial team review based on expected impact.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**Target / Proxy Definition:**
- **Primary Proxy Target:** `is_decaying` (Binary label: `1` if traffic is declining severely, `0` otherwise).

**Where the label comes from:**
- Since we do not have a pre-existing human label for "needs refresh", we construct a **historically observed proxy label** directly from traffic performance data.
- **Defined Logic:** An article is labeled `1` (`is_decaying`) if its `trend_direction == 'down'` **AND** its recent traffic drop exceeds a baseline threshold (e.g., `trend_pct < -20%` or significant click drop from previous 30 days to last 30 days: `clicks_last_30d < clicks_prev_30d`).
- This label is an **observed outcome-based proxy** (derived from historical performance drift) rather than an arbitrary manual heuristic, grounding the model in measurable traffic decay.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

**Primary Evaluation Metric:** Precision@K (e.g., Precision@Top 20%) & ROC-AUC

**What number means 'good'?:**
- **Primary Operational Metric (Precision@Top K):** Precision@Top 20% $\ge 0.75$ (75%+). In practice, content teams have limited editorial bandwidth and can only refresh a subset of articles per month (e.g., top 100 or top 20%). A high Precision@K guarantees that at least 75% of the articles flagged by the model for refresh are truly high-priority decaying assets, avoiding wasted editorial effort.
- **Secondary Offline Metric (ROC-AUC):** ROC-AUC $\ge 0.80$. Demonstrates that the model effectively ranks decaying articles higher than healthy articles across all probability thresholds.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** **One row = One unique published article** (`content_id`), owned by a specific client (`client_id`), captured at a single historical evaluation window (90-day performance snapshot).

**DataFrame Specification:**
- The slice below displays the core grain of the dataset: identifying features (`content_id`, `client_id`), metadata (`content_type`, `freshness_tier`), engagement metrics (`clicks_last_30d`, `impressions_last_30d`), and the engineered binary target (`target_is_decaying`).

In [1]:
!git clone https://github.com/PTD504/flyrank-ai-ml-internship

Cloning into 'flyrank-ai-ml-internship'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 128 (delta 42), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.85 MiB | 25.23 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [2]:
import pandas as pd

# Load starter data
data_dir = "flyrank-ai-ml-internship/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_dir)

# Construct target proxy column (1 = decaying & lost clicks, 0 = stable/growing)
df['target_is_decaying'] = ((df['trend_direction'] == 'down') & (df['clicks_last_30d'] < df['clicks_prev_30d'])).astype(int)

# Display unit of analysis slice (1 row = 1 article)
slice_df = df[['content_id', 'client_id', 'content_type', 'freshness_tier', 'clicks_last_30d', 'clicks_prev_30d', 'target_is_decaying']].head(5)
print(f"Unit of Analysis Verified: One row = One article (Shape: {df.shape[0]:,} articles x {df.shape[1]} columns)")
slice_df

Unit of Analysis Verified: One row = One article (Shape: 30,000 articles x 45 columns)


,content_id,client_id,content_type,freshness_tier,clicks_last_30d,clicks_prev_30d,target_is_decaying
0,content_304f48230142,client_f369cb89fc,keyword article,0-30,2,13,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0-30,2,1,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0-30,1,3,1
3,content_331d6c4de07b,client_19581e27de,keyword article,0-30,22,17,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0-30,10,2,0


## 5. Why ML beats a fixed rule here

**Why an if-statement fails (Why ML wins):**
- **Multi-dimensional non-linear signals:** A simple heuristic rule (e.g., `if content_age > 180 days and clicks dropped by 20% -> refresh`) fails because content decay is non-linear and context-dependent. A 180-day-old article in a fast-moving topic might be obsolete, while a 180-day-old evergreen piece might perform exceptionally well.
- **High-dimensional interaction (44 features):** Performance drift depends on complex co-occurrences of signals: impression velocity, CTR shifts, search intent changes, competition levels, engagement rates, and emerging AI traffic percentage. Manual `if/else` rules cannot scale across 30,000 articles with varying content types without introducing massive false positives.
- **Continuous Priority Scoring vs. Hard Cutoffs:** Hard rules create binary arbitrary cutoffs, treating an article with a 19.9% drop completely differently from one with a 20.1% drop. ML produces a calibrated continuous risk score ($0.0$ to $1.0$), dynamically adapting queue priorities to match available content rewrite bandwidth.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.